### Function objects are closures

In [2]:
def great():
    print("Hello")
    
    
say_hi = great

say_hi()

Hello


In [3]:
def apply_twice(func, value):
    return func(func(value))


def add_one(x):
    return x + 1


print(apply_twice(add_one, 5))

7


In [4]:
def get_operation():
    
    def multiply(a, b):
        return a * b
    
    return multiply

operation = get_operation()
print(operation(5, 6))

30


In [5]:
def shout():
    return "HI"


x = shout
y = shout()

print(type(x)) # return function object callable object
print(type(y)) # return string object

<class 'function'>
<class 'str'>


### Closures

**Closure** is what happens when a function is defined inside another function and it uses a variable from that outer function

In [6]:
def make_multiplier(n):
    def multiplier(x):
        return x * n
    return multiplier


double = make_multiplier(2)
triple = make_multiplier(3)

print(double(5))  # Output: 10
print(triple(5))  # Output: 15

four_times = make_multiplier(4)
print(four_times(5))  # Output: 4 * 5 = 20

10
15
20


In [7]:
def make_counter():
    
    count = 0
    
    def increment():
        
        nonlocal count # if we remove that line, it will give an error because count is not defined in the local scope of increment function.
        
        count += 1
        return count
    
    return increment

counter = make_counter()


print(counter()) 
print(counter()) 
print(counter()) 


1
2
3


In [8]:
def outer():
    
    items = []
    
    def inner(item):
        items.append(item)
        return items

    return inner



add_to_list = outer()
print(add_to_list(1))
print(add_to_list(2))
print(add_to_list(3))

[1]
[1, 2]
[1, 2, 3]


In [9]:
"""
when we use nonlocal keyword, it allows us to modify the variable in the enclosing scope (in this case, the outer function) rather than creating a new local variable in the inner function. 
This is useful when we want to maintain state across multiple calls to the inner function.
"""

"""
we don't need to use nonlocal in the outer function because we are not modifying the items list itself, 
we are just appending to it. The list is mutable, so we can modify its contents without needing to declare it as nonlocal.
"""

"\nwe don't need to use nonlocal in the outer function because we are not modifying the items list itself, \nwe are just appending to it. The list is mutable, so we can modify its contents without needing to declare it as nonlocal.\n"

In [10]:
class Broken:
    pass # don't have __add__ method


def outer():
    
    x = Broken()
    
    def inner():
        
        nonlocal x # if we remove that line, it will give an error because x is not defined in the local scope of inner function.
                   # that's why we get UnboundLocalError: local variable 'x' referenced before assignment
                     
        x += 1  # This will raise an error because Broken does not support addition.
        return x

    return inner

add_to_broken = outer()


try:
    print(add_to_broken())
except TypeError as e:
    print(f"Error: {e}")  # Output: Error: unsupported operand type(s) for +=: 'Broken' and 'int'

Error: unsupported operand type(s) for +=: 'Broken' and 'int'


### Decorator


In [11]:
import time

def timer(func):
    
    def wrapper(*args, **kwargs):
        start_time = time.time()
        
        result = func(*args, **kwargs)
        
        end_time = time.time()
        
        print(f"Execution time: {end_time - start_time} seconds")
        return result
    
    return wrapper


@timer
def slow_function():
    time.sleep(2)
    return "Finished"

slow_function()


Execution time: 2.000114679336548 seconds


'Finished'

In [12]:
print(slow_function.__name__) # if we change name of wrapper function output will be changed

wrapper


In [13]:
def slow_add(a, b):
    time.sleep(2)
    return a + b

slow_add = timer(slow_add) # if we don't use decorator syntax, we can manually apply the decorator like this
slow_add(5, 3)

Execution time: 2.000089406967163 seconds


8

In [14]:
from functools import wraps


def timer(func):
    
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        
        result = func(*args, **kwargs)
        
        end_time = time.time()
        
        print(f"Execution time: {end_time - start_time} seconds")
        return result
    
    return wrapper


@timer
def slow_function():
    time.sleep(2)
    return "Finished"

print(slow_function())

print(slow_function.__name__) # now it will return slow_function instead of wrapper because we used @wraps decorator

Execution time: 2.0001206398010254 seconds
Finished
slow_function


## why we decorate wrapper function not timer function

because when we call


```
slow_add = timer(slow_add)
```

timer function returns wrapper thats why


In [15]:

from functools import wraps
import time

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):        
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"Execution time: {end_time - start_time} seconds")
        return result
    return wrapper


@timer
def slow_add(a, b):
    time.sleep(2)
    return a + b


print(slow_add(5, 3))
print(slow_add.__name__)

Execution time: 2.0000977516174316 seconds
8
slow_add


### Decorators with arguments


In [1]:
def retry(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):

            for _ in range(times):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"Error: {e}. Retrying...")

            raise Exception(f"Failed after {times} attempts.")
        return wrapper
    return decorator

In [3]:
# @times(3)
# def fake_api_call():
#     ...


# fake_api_call = times(3)(fake_api_call)

In [8]:
def code():

    counter = 3

    def inner():

        print(counter)

        counter -= 1  # This will raise an UnboundLocalError because counter is being assigned a new value in the local scope of inner, which makes it a local variable.
        return counter

    return inner


test = code()
test()

UnboundLocalError: cannot access local variable 'counter' where it is not associated with a value

In [45]:
from functools import wraps
import random


def times(n):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):

            for i in range(n):

                try:
                    result = func(*args, **kwargs)
                    return result
                except ValueError:
                    print(f"Value error in attempt {i + 1}")

            raise Exception("All attempts was unsuccessfull")

        return wrapper
    return decorator


@times(3)
def flaky_call():

    if random.random() < 0.7:
        raise ValueError("Value error")

    return "Success"



try:
    print(flaky_call())
except Exception:
    print("All attepts was unsuccessfull")

Value error in attempt 1
Value error in attempt 2
Value error in attempt 3
All attepts was unsuccessfull
